<a href="https://colab.research.google.com/github/kofisarf/Plant-Disease-Detection---Group-9/blob/CODE/Plant_Disease_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q kaggle

In [2]:
from google.colab import files
files.upload()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_673/1613494533.py", line 2, in <cell line: 0>
    files.upload()
  File "/usr/local/lib/python3.12/dist-packages/google/colab/files.py", line 69, in upload
    uploaded_files = _upload_files(multiple=True)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/files.py", line 161, in _upload_files
    result = _output.eval_js(
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/output/_js.py", line 40, in eval_js
    return _message.read_reply_from_input(request_id, timeout_sec)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_message.py", line 96, in read_reply_from_input
    time.sleep(0.025)
Keybo

TypeError: object of type 'NoneType' has no len()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plantvillage_data


In [ ]:
import os
import shutil

source_dir = 'plantvillage_data/plantvillage dataset/color'

Crop_data = 'Crop_data'

os.makedirs(Crop_data, exist_ok=True)

Study_crops = ['Tomato', 'Corn_(maize)', 'Potato', 'Pepper']

for folder_name in os.listdir(source_dir):
    if any(crop in folder_name for crop in Study_crops):

        src_path = os.path.join(source_dir, folder_name)
        dst_path = os.path.join(Crop_data, folder_name)

        if not os.path.exists(dst_path):
            shutil.copytree(src_path, dst_path)
            print(f"Copied: {folder_name}")

print("Data filtering complete!")

In [ ]:
os.listdir("Crop_data")

In [ ]:
# split-folders helper
!pip install split-folders

import splitfolders

# Split your filtered dataset into train (80%), val (10%), test (10%)
splitfolders.ratio(
    "Crop_data",       # Your filtered folder name
    output="data_split",      # New folder with split subdirectories
    seed=42,
    ratio=(0.8, 0.1, 0.1)
)

print("✅ Data successfully split into train, val, and test folders!")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Rescale image pixels (from 0-255 down to 0.0-1.0)
datagen = ImageDataGenerator(rescale=1./255)

IMG_SIZE = (128, 128) # Resizing images to 128x128 keeps training fast for Thursday's demo
BATCH_SIZE = 32

# Create the training stream
train_generator = datagen.flow_from_directory(
    'data_split/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Create the validation stream
val_generator = datagen.flow_from_directory(
    'data_split/val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Create the testing stream
test_generator = datagen.flow_from_directory(
    'data_split/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Get the number of crop classes automatically
num_classes = train_generator.num_classes

# Build a simple baseline CNN architecture
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax') # Outputs probability for each crop disease class
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model for 5 epochs to establish a quick baseline
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/crop_disease_model.keras')

In [3]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 19)             │         2,451 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,184,315 (84.63 MB)

 Trainable params: 7,394,771 (28.21 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 14,789,544 (56.42 MB)